In [9]:
import cv2
import pytesseract
from PIL import Image
import json
import os

读取坐标信息

In [10]:
refPts_path = r"D:\Intelligent_Image_Recognition\ProcessingofFixed_Information\refPts\refPts2.json"

with open(refPts_path, 'r') as f:
    refPts = json.load(f)

# 打印 refPts
for i, pts in enumerate(refPts):
    print(f"Selected Region {i+1}: Top-left: {pts[0]}, Bottom-right: {pts[1]}")

Selected Region 1: Top-left: [326, 3060], Bottom-right: [1330, 3422]
Selected Region 2: Top-left: [1326, 3062], Bottom-right: [1544, 3184]
Selected Region 3: Top-left: [1332, 3184], Bottom-right: [1544, 3300]
Selected Region 4: Top-left: [1332, 3306], Bottom-right: [1540, 3410]
Selected Region 5: Top-left: [1544, 3062], Bottom-right: [2180, 3184]
Selected Region 6: Top-left: [1544, 3184], Bottom-right: [2182, 3302]
Selected Region 7: Top-left: [1540, 3302], Bottom-right: [1752, 3414]
Selected Region 8: Top-left: [1752, 3302], Bottom-right: [1968, 3418]
Selected Region 9: Top-left: [1970, 3302], Bottom-right: [2180, 3418]


输入数据并处理


In [11]:
pytesseract.pytesseract.tesseract_cmd =r"D:\tesseract\tesseract.exe"
img_path = r"D:\Intelligent_Image_Recognition\pdf_img\results\res1\page_1.png"

def preprocess_image(image_path):
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, binary = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)
    return binary

img = preprocess_image(img_path)
img = cv2.cvtColor(img,cv2.COLOR_BGR2RGB)

字符的识别

In [12]:
text = pytesseract.image_to_string(img,lang='chi_sim')
print(text)

图 纸 目 录

名                                    ，                                       图号                         图名

图纸目未                      1             1                                          -603                地面桥 小箱洪错下螺旋入大样图
施工图设计总说明                        ;                 1                                                       -701                   地面桥 小箱粱支座及预埋件构造图
主要工程数量表                                                                                -8                  地面桥 小箱深支座扫石钢和图
主线 桥位平面                    1             1                                           -901                地面桥 小箱深堵头板构造钢艇图

地面桥 桥位平面布置图                                                                    001 ~ 002                 预制小箱粱典型断面图

主线 总体立面布置图                                                                 1~122               主线 制小箱梁布染图

囊遗 总体立面布置图             1                                         -123~126                C/H正道 预制小箱梁布梁国

高架桥 总体平面布置图                                  

主程序

In [14]:

W,H,C = img.shape
output_image = img.copy()


    # 创建一个目录来保存 Box 文件
box_path = r"D:\Intelligent_Image_Recognition\ProcessingofFixed_Information\boxes\box2"
os.makedirs(box_path, exist_ok=True)

#读取框选坐标信息
for i, (top_left, bottom_right) in enumerate(refPts):
    x1, y1 = top_left
    x2, y2 = bottom_right
    
    # 裁剪出当前矩形区域
    roi = img[y1:y2, x1:x2]

    # 使用 Tesseract 进行文字识别
    boxes = pytesseract.image_to_boxes(roi, lang='chi_sim')
    
     # 存储当前 Box 的信息
    box_info = []
    
    for b in boxes.splitlines():
        b = b.split(" ")
        #print(f"Raw box data: {b}")  # 打印原始的 box 数据
        print(f"Box {i+1}: {b}")
        if len(b) == 6:  # 确保 b 包含字符和坐标信息
            character = b[0]
            #print(character)
            try:
                bx1, by1, bx2, by2 = int(b[1]), int(b[2]), int(b[3]), int(b[4])

                #转换坐标到原图上的位置
                bx1 += x1
                bx2 += x1
                by1 = H - (by1 + y1)
                by2 = H - (by2 + y1)

         # 存储识别到的字符信息
                box_info.append({
                    "character": character,
                    "bbox": [bx1, by1, bx2, by2]
                })

            except ValueError:
                print(f"Invalid coordinate values in box data: {b}")

     # 保存当前 Box 的信息到 JSON 文件
    box_file_path = os.path.join(box_path, f'box_{i+1}.json')
    with open(box_file_path, 'w',encoding='utf-8') as f:
        json.dump(box_info, f, ensure_ascii=False, indent=4)   

        
        #cv2.rectangle(output_image, (bx1, by1), (bx2, by2), (0, 0, 255), 1)
        #cv2.putText(output_image, b[0], (bx1, by1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)

# 显示带有识别结果的图像
#cv2.imshow("text", output_image)
#cv2.waitKey(0)
#cv2.destroyAllWindows()


Box 1: ['同', '294', '237', '343', '290', '0']
Box 1: ['济', '353', '235', '408', '291', '0']
Box 1: ['大', '438', '236', '469', '291', '0']
Box 1: ['学', '495', '236', '530', '292', '0']
Box 1: ['建', '561', '236', '655', '292', '0']
Box 1: ['筑', '660', '238', '681', '289', '0']
Box 1: ['设', '684', '237', '745', '290', '0']
Box 1: ['计', '740', '236', '803', '292', '0']
Box 1: ['研', '802', '236', '840', '288', '0']
Box 1: ['究', '846', '237', '900', '291', '0']
Box 1: ['院', '925', '236', '962', '291', '0']
Box 1: ['《', '393', '156', '415', '215', '0']
Box 1: ['集', '434', '159', '547', '215', '0']
Box 1: ['团', '565', '157', '588', '215', '0']
Box 1: [')', '588', '136', '615', '220', '0']
Box 1: ['有', '641', '160', '694', '216', '0']
Box 1: ['限', '701', '160', '733', '211', '0']
Box 1: ['公', '756', '160', '796', '213', '0']
Box 1: ['司', '804', '160', '851', '211', '0']
Box 1: ['T', '277', '107', '292', '135', '0']
Box 1: ['O', '294', '107', '310', '135', '0']
Box 1: ['N', '313', '107', '330', 

打印信息

In [15]:

def process_json_files(folder_path):
    # 获取文件夹中的所有 JSON 文件
    files = [f for f in os.listdir(folder_path) if f.endswith('.json')]
    
    for file_name in files:
        file_path = os.path.join(folder_path, file_name)
        with open(file_path, 'r', encoding='utf-8') as file:
            data = json.load(file)
            # 假设每个 JSON 文件中的数据是一个字符信息的列表
            combined_string = ''.join(item['character'] for item in data)
            print(f"内容来自文件: {file_name}")
            print(combined_string)
            print('-' * 40)  # 分隔线

# 使用示例
folder_path = box_path
process_json_files(folder_path)


内容来自文件: box_1.json
同济大学建筑设计研究院《集团)有限公司TONGJIARCHITECTURALDESIGN(Group)Co.,Ltd，月从设计TJAD~
----------------------------------------
内容来自文件: box_2.json
项目名称ProjectName
----------------------------------------
内容来自文件: box_3.json
子项名称Sub-Project
----------------------------------------
内容来自文件: box_4.json
项目编号ProjectNo，
----------------------------------------
内容来自文件: box_5.json
南阳市中心城区外环中综合开发项目设计~
----------------------------------------
内容来自文件: box_6.json
04段冯楼北〈南都路-龙祥路)城市道路工程圳~
----------------------------------------
内容来自文件: box_7.json
22Z-BB-096
----------------------------------------
内容来自文件: box_8.json
子项编号Sub-ProjectNo.
----------------------------------------
内容来自文件: box_9.json

----------------------------------------
